# 02 - Family group rule ("woman-child-group")

**Idea:** start from the gender rule, then fix its two biggest mistakes using **families**.

Families mostly lived or died together. Many families have members in *both* train and test.
So if we know what happened to the rest of a family, we can make a better guess.

**The rule**
1. Everyone starts with the gender rule: women live, men die.
2. **Boys** (title "Master") → **live** if every woman and child in their family survived.
3. **Females** → **die** if every woman and child in their family died.
4. Adult men are left out of families. They mostly died no matter what.

Output: `submission_v2_group.csv` (the old `submission.csv` is kept for comparison).

In [1]:
import pandas as pd

train = pd.read_csv("titanic/train.csv")
test = pd.read_csv("titanic/test.csv")

# Combine both files: families are split across train and test
df = pd.concat([train, test], ignore_index=True)
is_train = df.Survived.notna()
print(len(df), "passengers total")

1309 passengers total


## Step 1: Who is a woman or child?
Female, or a boy with the title "Master".

In [2]:
df["Title"] = df.Name.str.extract(r",\s*([^\.]+)\.")[0]
df["IsWomanOrChild"] = (df.Sex == "female") | (df.Title == "Master")

# Why boys matter: they are the males who survive most
t = df[is_train]
t.groupby(t.Title.where(t.Title.isin(["Mr", "Mrs", "Miss", "Master"]), "Other")).Survived.agg(["mean", "count"]).round(2)

,mean,count
Title,,
Master,0.57,40
Miss,0.70,182
Mr,0.16,517
Mrs,0.79,125
Other,0.44,27


## Step 2: Build family groups
**Group key:** surname + class + ticket (without its last digit) + fare + port.

This finds people who share a name *and* travelled on the same booking.
Adult men and people with no family member are put in a group called `none`.

In [3]:
df["Surname"] = df.Name.str.split(",").str[0]
df["GroupId"] = (df.Surname + "-" + df.Pclass.astype(str) + "-" + df.Ticket.str[:-1]
                 + "-" + df.Fare.astype(str) + "-" + df.Embarked.fillna(""))
df.loc[~df.IsWomanOrChild, "GroupId"] = "none"
df.loc[df.groupby("GroupId").GroupId.transform("size") < 2, "GroupId"] = "none"

groups = df[df.GroupId != "none"]
print("Family groups found:", groups.GroupId.nunique())
print("Women/children in a group:", len(groups))

Family groups found: 80
Women/children in a group: 219


## Step 3: Families really do live or die together
For each group in train, what share of its women and children survived?
Most groups are all 0% or all 100%.

In [4]:
rates = df[is_train & (df.GroupId != "none")].groupby("GroupId").Survived.mean()
pd.cut(rates, [-0.01, 0.0, 0.99, 1.0], labels=["all died", "mixed", "all lived"]).value_counts()

Survived
all lived    53
all died     21
mixed         2
Name: count, dtype: int64

## Step 4: The rule as a function
`leave_one_out=True` hides each passenger's own answer when judging their family.
Without this, the check in Step 5 would cheat and look far too good.

In [5]:
def predict(rows, labelled, leave_one_out=False):
    stats = labelled[labelled.GroupId != "none"].groupby("GroupId").Survived.agg(["sum", "count"])
    pred = (rows.Sex == "female").astype(int)          # start: gender rule
    for i, r in rows.iterrows():
        if r.GroupId not in stats.index:
            continue                                     # no known family -> keep gender rule
        s, c = stats.loc[r.GroupId]
        if leave_one_out:
            s, c = s - r.Survived, c - 1
        if c == 0:
            continue
        rate = s / c
        if r.Title == "Master" and rate == 1:
            pred[i] = 1                                  # boy whose family all lived
        elif r.Sex == "female" and rate == 0:
            pred[i] = 0                                  # female whose family all died
    return pred

## Step 5: Check on train (leave-one-out)
Note: cross-validation was too hopeful last time (0.83 vs Kaggle 0.76). Treat this as a rough guide.

In [6]:
trn = df[is_train]
gender_acc = ((trn.Sex == "female").astype(int) == trn.Survived).mean()
group_acc = (predict(trn, trn, leave_one_out=True) == trn.Survived).mean()
print(f"Gender rule      : {gender_acc:.3f}")
print(f"Family group rule: {group_acc:.3f}")

Gender rule      : 0.787
Family group rule: 0.846


## Step 6: Predict the test set

In [7]:
tst = df[~is_train]
pred = predict(tst, trn)

sub = pd.DataFrame({"PassengerId": tst.PassengerId.astype(int), "Survived": pred.astype(int).values})
sub.to_csv("submission_v2_group.csv", index=False)
print(len(sub), "rows,", f"{sub.Survived.mean():.0%} predicted survived")

418

 rows, 36% predicted survived


## Step 7: Sanity check — what changed?
Only a few passengers should differ from the gender rule. A big number would mean a bug.

In [8]:
gender = pd.read_csv("titanic/gender_submission.csv")
v1 = pd.read_csv("submission.csv")
print("Different from gender rule :", (sub.Survived.values != gender.Survived.values).sum())
print("Different from v1 (logreg) :", (sub.Survived.values != v1.Survived.values).sum())

flipped = sub.Survived.values != gender.Survived.values
tst[flipped][["PassengerId", "Name", "Sex", "Pclass"]].assign(Prediction=sub.Survived.values[flipped])

Different from gender rule : 16
Different from v1 (logreg) : 23


,PassengerId,Name,Sex,Pclass,Prediction
909,910,"Ilmakangas, Miss. Ida Livija",female,3,0
924,925,"Johnston, Mrs. Andrew G (Elizabeth Lily"" Watson)""",female,3,0
928,929,"Cacic, Miss. Manda",female,3,0
955,956,"Ryerson, Master. John Borie",male,1,1
980,981,"Wells, Master. Ralph Lester",male,2,1
1023,1024,"Lefebre, Mrs. Frank (Frances)",female,3,0
1031,1032,"Goodwin, Miss. Jessie Allis",female,3,0
1052,1053,"Touma, Master. Georges Youssef",male,3,1
1079,1080,"Sage, Miss. Ada",female,3,0
1085,1086,"Drew, Master. Marshall Brines",male,2,1


## Summary
- Boys whose families lived are now predicted to **live**.
- Women whose families died are now predicted to **die**.
- Everyone else uses the gender rule.
- This kind of family rule usually scores around **0.80–0.81** on Kaggle, but that's not guaranteed.